# Bronze Kafka Validation

This notebook checks the unified Bronze Delta table at `s3a://bronze/kafka`
and validates that Kafka events are landing as expected.


## Imports and Spark session
Build a Spark session with S3A + Delta settings for MinIO-backed storage.


In [1]:
import os
from pyspark.sql import SparkSession


In [2]:
def build_spark():
    """Create a SparkSession configured for MinIO (S3A) and Delta."""
    app_name = os.getenv('SPARK_APP_NAME', 'bronze-kafka-check')
    return (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.delta.logStore.class', 'org.apache.spark.sql.delta.storage.S3SingleDriverLogStore')
        .config('spark.hadoop.fs.s3a.endpoint', os.getenv('DATA_LAKE_ENDPOINT', 'http://minio:9000'))
        .config('spark.hadoop.fs.s3a.access.key', os.getenv('DATA_LAKE_ACCESS_KEY_ID', 'minioadmin'))
        .config('spark.hadoop.fs.s3a.secret.key', os.getenv('DATA_LAKE_SECRET_ACCESS_KEY', 'minioadmin'))
        .config('spark.hadoop.fs.s3a.path.style.access', 'true')
        .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
        .getOrCreate()
    )


## Bronze readers
Helper functions for loading and inspecting the Bronze Delta table.


In [3]:
def get_bronze_path():
    """Resolve Bronze path from env or default."""
    return os.getenv('BRONZE_STREAM_PATH', 's3a://bronze/kafka')

def load_bronze_df(spark):
    """Load the Bronze Delta table into a DataFrame."""
    return spark.read.format('delta').load(get_bronze_path())

def show_samples(df, limit=10):
    """Display a sample of Bronze records."""
    df.show(limit, truncate=False)

def show_counts(df):
    """Show total record count."""
    print(f'bronze count: {df.count()}')

def show_record_type_counts(df):
    """Show counts by record_type."""
    (df.groupBy('record_type').count().orderBy('record_type')).show(truncate=False)

def show_kafka_partition_counts(df):
    """Show counts by Kafka partition."""
    (df.groupBy('kafka_partition').count().orderBy('kafka_partition')).show(truncate=False)

def show_file_counts(df, limit=20):
    """Show record counts per Delta file."""
    (df.groupBy('_metadata.file_path')
       .count()
       .orderBy('count', ascending=False)
       .show(limit, truncate=False))


## Run the checks
Start Spark, load Bronze, and inspect counts/samples.


In [4]:
spark = build_spark()
print(f'Spark version: {spark.version}')


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/08 08:41:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.0.1


In [7]:
bronze_df = load_bronze_df(spark)


In [8]:
show_counts(bronze_df)


26/01/08 08:44:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

bronze count: 4


In [9]:
show_record_type_counts(bronze_df)


+-----------+-----+
|record_type|count|
+-----------+-----+
|business   |2    |
|user       |2    |
+-----------+-----+



In [ ]:
show_kafka_partition_counts(bronze_df)


[Stage 20:>                                                         (0 + 1) / 1]

In [ ]:
show_samples(bronze_df, limit=5)


## File-level counts
Check how many records landed in each Delta file.


In [ ]:
show_file_counts(bronze_df, limit=20)
